In [ ]:
!pip install nselib

In [2]:
import nselib
from nselib import capital_market
from nselib import derivatives


In [3]:
import numpy as np
from scipy.stats import norm

def greeks(
    S,          # Spot price
    K,          # Strike price
    T_days,     # Time to expiry in days
    IV,         # Implied Volatility (e.g. 0.16 for 16%)
    r=0.06,     # Risk-free rate (India ~6%)
    option="call"
):
    # Convert days to years
    T = T_days / 365.0
    sigma = IV

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option.lower() == "call":
        delta = norm.cdf(d1)
        theta = (
            - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
            - r * K * np.exp(-r * T) * norm.cdf(d2)
        )
    else:
        delta = norm.cdf(d1) - 1
        theta = (
            - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
            + r * K * np.exp(-r * T) * norm.cdf(-d2)
        )

    # Theta per day (broker convention)
    theta_per_day = theta / 365

    return delta, theta_per_day

In [4]:
import numpy as np # Ensure numpy is imported for vectorized operations
from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma):
    # Calculate d1 and d2 using numpy functions for vectorization
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Call price using numpy functions
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

    return call_price # Removed round to allow for vectorized output


def black_scholes_put(S, K, T, r, sigma):
    # Calculate d1 and d2 using numpy functions for vectorization
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # Put price using numpy functions
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return put_price # Removed round to allow for vectorized output


In [5]:
Symbol='NIFTY'
expiry__date=str(input('Enter expiry date, example :13-04-2026 or 21-04-2026, etc'))
data = derivatives.nse_live_option_chain(Symbol,expiry__date)[['CALLS_IV','Strike_Price','PUTS_IV']]

Enter expiry date, example :13-04-2026 or 21-04-2026, etc13-04-2026


In [6]:
data

,CALLS_IV,Strike_Price,PUTS_IV
0,0.00,20100,46.49
1,0.00,20150,45.60
2,49.79,20200,44.98
3,39.87,20250,44.96
4,48.73,20300,44.32
...,...,...,...
114,31.94,25800,0.00
115,31.48,25850,40.86
116,32.44,25900,0.00
117,33.98,25950,41.74


In [7]:
spot_price = capital_market.index_data('NIFTY 50', period = '1d')['CLOSE_INDEX_VAL'][0]
print(spot_price)

22968.25


In [8]:
from datetime import date
import pandas as pd

current_date = date.today()
expiry_date_str = expiry__date  # Get expiry date string from previous cell
expiry_date = pd.to_datetime(expiry_date_str, format='%d-%m-%Y').date()

delta = expiry_date - current_date
T = delta.days / 365.0

In [ ]:
call_ltp_values = []
put_ltp_values = []

for index, row in data.iterrows():
    K = row['Strike_Price']
    call_iv = row['CALLS_IV'] / 100
    put_iv = row['PUTS_IV'] / 100

    call_price = black_scholes_call(spot_price, K, T, 0.06, call_iv)
    put_price = black_scholes_put(spot_price, K, T, 0.06, put_iv)

    call_ltp_values.append(call_price)
    put_ltp_values.append(put_price)

data['CALL_LTP'] = call_ltp_values
data['PUT_LTP'] = put_ltp_values

In [ ]:
put_delta =[]
call_delta = []
thetaB = []
for i in range(len(data)):
  S = int(spot_price)
  K = int(data['Strike_Price'][i])
  IV = float(data['CALLS_IV'][i]/100)
  delta, theta = greeks(S, K, T, IV, option="call")
  delta1, theta1 = greeks(S, K, T, IV, option="put")
  call_delta.append(float(round(delta, 2)))
  thetaB.append(float(round(theta, 2)))
  put_delta.append(float(round(-delta1, 2)))
data['call_delta'] = call_delta
data['put_delta'] = put_delta
data['thetaB'] = thetaB

In [11]:
T*365

7.0

In [12]:
data[(data['Strike_Price']<spot_price+500) & (data['Strike_Price']>spot_price-500)]

,CALLS_IV,Strike_Price,PUTS_IV,CALL_LTP,PUT_LTP,call_delta,put_delta,thetaB
48,32.96,22500,33.66,706.851384,220.585980,1.00,-0.00,-3.70
49,32.36,22550,33.33,666.715877,233.645690,1.00,0.00,-3.71
50,31.72,22600,32.93,626.733456,246.613898,1.00,0.00,-3.72
51,32.32,22650,33.02,602.113929,266.151663,1.00,0.00,-3.72
52,31.99,22700,32.79,567.448385,282.768098,1.00,0.00,-3.73
53,31.13,22750,32.20,527.129063,295.821981,1.00,0.00,-3.81
54,31.14,22800,32.17,498.301533,316.599956,1.00,0.00,-6.44
55,31.41,22850,32.00,473.727585,336.584182,0.99,0.01,-45.56
56,30.77,22900,31.96,438.736970,359.149267,0.91,0.09,-222.89
57,30.79,22950,31.70,413.024554,379.903280,0.64,0.36,-503.22


In [19]:
import matplotlib.pyplot as plt # Import matplotlib for plotting
import numpy as np # Ensure numpy is imported
import plotly.graph_objects as go # Import Plotly for interactive plots

cur=[]
exp=[]
stk=[]
tupe=[[],[],[]]
total_prem=0
n=int(input('enter no. of options:'))
for i in range(n):
  print('enter option',i+1)
  b=str(input('enter CALL or PUT:'))
  a=int(input('Enter index of option (see the perevious cell for index):'))
  qt=int(input('Enter quantity:'))
  tupe[0].append(b)
  tupe[1].append(a)
  tupe[2].append(qt)

  # Ensure k3, k1, k2 are floats for consistency in calculations
  k3_call = float(data['CALLS_IV'][a])
  k3_put = float(data['PUTS_IV'][a]) # Get PUTS_IV as well
  k1=float(data['Strike_Price'][a])
  k2=float(spot_price)
  premium = float(data['CALL_LTP'][a]) if b.upper()=='CALL' else float(data['PUT_LTP'][a])
  total_prem = total_prem+premium
  # Define the range of spot prices using numpy
  # Adjust the range to be slightly wider than the original -15 to 15*50 to ensure good visual spread
  price_range = np.arange(k2 - 20*50, k2 + 21*50, 20)

  option_prices = np.array([])
  option_type_label = ''
  exp_payoff = np.zeros_like(price_range, dtype=float)

  # Calculate option prices using the vectorized Black-Scholes function
  if b.upper()=='CALL':
    option_prices = black_scholes_call(price_range, k1, T, 0.06, k3_call/100)
    option_type_label = 'Call Option'
    exp_payoff = qt * (np.maximum(price_range - k1, 0) - premium)
  elif b.upper()=='PUT':
    option_prices = black_scholes_put(price_range, k1, T, 0.06, k3_put/100)
    option_type_label = 'Put Option'
    exp_payoff = qt * (np.maximum(k1 - price_range, 0) - premium)
  else:
    print("Invalid option type entered. Please enter 'CALL' or 'PUT'.")

  curr_payoff = (qt*(option_prices-premium))
  cur.append(curr_payoff)
  exp.append(exp_payoff)
  stk.append(k1)


# Plotting the graph
if len(exp) > 0: # Ensure at least one option was entered
    expt = np.sum(np.array(exp), axis=0)
    curt = np.sum(np.array(cur), axis=0)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=price_range, y=curt, mode='lines', name='Current Payoff', line=dict(color='blue')))
    fig.add_trace(go.Scatter(x=price_range, y=expt, mode='lines', name='Expected Payoff', line=dict(color='orange')))

    # Loop through strike prices to add vertical lines
    for j, strike_price in enumerate(stk):
        fig.add_vline(x=strike_price, line_dash="dash", line_color="red", annotation_text=f'Strike Price{j+1}')

    fig.add_vline(x=k2, line_dash="dot", line_color="green", annotation_text=f'Spot Price')
    fig.add_hline(y=0, line_color='Black',line_width=1)
    fig.update_layout(
        title='Option Payoff Strategy',
        xaxis_title='Price of asset',
        yaxis_title='Payoff',
        hovermode='x unified' # Provides a nice tooltip
    )

    fig.show()
else:
    print("No options were entered to plot.")

enter no. of options:2
enter option 1
enter CALL or PUT:CALL
Enter index of option (see the perevious cell for index):53
Enter quantity:10
enter option 2
enter CALL or PUT:PUT
Enter index of option (see the perevious cell for index):53
Enter quantity:10


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

def calculate_and_display_pnl(spot_val, t_val, iv_val):
    clear_output(wait=True)
    prct = []
    payoff_total = 0

    # Ensure t_val is not zero to avoid division by zero
    if t_val == 0:
        print("Time to expiry (t) cannot be zero. Please adjust the slider.")
        return

    # Assuming 'tupe', 'stk', 'data', 'black_scholes_call', 'black_scholes_put' are available globally
    # 'tupe' is a list of lists: [['CALL', 'PUT'], [45, 53], [1, 1]]
    # 'stk' is a list of strike prices for each option
    # Correction: Iterate through the options stored in 'tupe' correctly

    print(f"Current Parameters: Spot={spot_val}, Days to Expiry={t_val}, Implied Volatility={iv_val:.2f}")
    print("------------------------------------------------------------------")

    for i in range(len(tupe[0])):
        option_type = tupe[0][i]
        option_index_in_data = tupe[1][i] # Original index in 'data' DataFrame
        quantity = tupe[2][i]
        strike_price = stk[i] # Get the strike price for the current option from the 'stk' list

        price = 0.0
        original_premium = 0.0 # This is the premium at the time of strategy definition

        if option_type == 'CALL':
            price = black_scholes_call(spot_val, strike_price, t_val/365, 0.06, iv_val)
            original_premium = float(data['CALL_LTP'][option_index_in_data])
        elif option_type == 'PUT':
            price = black_scholes_put(spot_val, strike_price, t_val/365, 0.06, iv_val)
            original_premium = float(data['PUT_LTP'][option_index_in_data])

        # Payoff calculation: (current calculated price - original premium) * quantity
        pay = quantity * (price - original_premium)

        prct.append(price)
        payoff_total += pay

        print(f"Option {i+1} ({option_type} K={strike_price}, Qty={quantity}):")
        print(f"  Calculated Price: {price:.2f}")
        print(f"  Original Premium: {original_premium:.2f}")
        print(f"  P/L for this option: {pay:.2f}\n")

    print(f"Total Strategy P/L: {payoff_total:.2f}")

# --- Create Sliders ---

# Initial values (adjust as needed)
initial_spot = spot_price # From current notebook state
initial_t_days = T*365 # Example, assuming 7 days
initial_iv = 0.20 # Example, 20% implied volatility

spot_slider = widgets.FloatSlider(
    value=initial_spot,
    min=initial_spot - 1000,
    max=initial_spot + 1000,
    step=10,
    description='Spot Price:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)

t_slider_days = widgets.IntSlider(
    value=initial_t_days,
    min=1,
    max=T*365, # Max 90 days for typical short-term options
    step=1,
    description='Days to Exp.:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
)

iv_slider = widgets.FloatSlider(
    value=initial_iv,
    min=0.01,
    max=0.50,
    step=0.01,
    description='Implied Volatility:',
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)

# Link sliders to the function
interactive_output = widgets.interactive(calculate_and_display_pnl,
                                         spot_val=spot_slider,
                                         t_val=t_slider_days,
                                         iv_val=iv_slider)

display(interactive_output)
